# Джем MyIndie LVL 9

# 0. Подготовка

In [ ]:
import os
from datetime import datetime
# from pathlib import Path
# import shutil
import json
import requests
import re
import pandas as pd
import numpy as np


# Константы
MYINDIE_JAM_URL = "https://myindie.ru/jams/jam/myindie-game-jam-level-9"
MYINDIE_JAM_URL_PAGES = 3  # Количество страниц с играми на геймджеме
MYINDIE_JAM_SKIP_DOWNLOAD = True
OUTPUT_DIR = "output/"
CRITERIAS = ["art", "sound", "theme", "gameplay", "narrative", "overall_impression"]  # Список критериев для анализа

# Директория для html-файлов джема
jam_path = MYINDIE_JAM_URL.split('/')[-1]  # получаем имя джема вроде "myindie-game-jam-level-9"
if jam_path is None or jam_path == "":
    raise ValueError("Не удалось извлечь путь джема из URL. Проверьте правильность MYINDIE_JAM_URL.")
os.makedirs(os.path.join(OUTPUT_DIR, jam_path), exist_ok=True)

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_rows', None)


def get_jam_games_urls(jam_url):
    """ Скачивает страницу списка игр джема и извлекает URL игр. """

    response = requests.get(jam_url)

    if response.status_code != 200:
        print(f"Ошибка при получении страницы джема: `{response.status_code}`")
        return []

    games_urls = re.findall(r'/games/game/[\w-]+', response.text)
    games_urls = [f"https://myindie.ru{url}" for url in games_urls]
    print(f"Найдено {len(games_urls)} URL игр в: {jam_url}")

    return games_urls


def unflatten_nuxt_data(data):
    """ Разворачивает плоскую структуру данных Nuxt.js в дерево. """
    if not isinstance(data, list) or not data: return data
    memo = {}
    def resolve(val):
        if isinstance(val, int) and 0 <= val < len(data):
            if val not in memo:
                memo[val] = resolve_item(data[val])
            return memo[val]
        return val
    def resolve_item(item):
        if isinstance(item, dict):
            return {k: resolve(v) for k, v in item.items()}
        if isinstance(item, list):
            return [resolve(v) for v in item]
        return item
    return resolve_item(data[1])

# 1. Скачиваем все страницы игр геймджема

1.1 Ищем все игры джема

In [36]:
if not MYINDIE_JAM_SKIP_DOWNLOAD:
    games_urls = []
    for page in range(1, MYINDIE_JAM_URL_PAGES + 1):
        paged_url = f"{MYINDIE_JAM_URL}/games?page={page}"
        print(f"Обрабатываем страницу: {paged_url}")
        urls = get_jam_games_urls(paged_url)
        games_urls.extend(urls)

    print(f"\nНайдено {len(games_urls)} игр на {MYINDIE_JAM_URL_PAGES} страницах:\n{chr(10).join(games_urls)}")

1.2 Скачать все HTML-страницы игр геймджема

In [37]:
if not MYINDIE_JAM_SKIP_DOWNLOAD:
    jam_games_file_path = os.path.join(OUTPUT_DIR, jam_path, f"jam_games.txt")
    if os.path.exists(jam_games_file_path):
        os.remove(jam_games_file_path)
    jam_games_file = open(jam_games_file_path, 'a', encoding='utf-8')

    # DEBUG_GAMES_MAX = 2  # delete
    for i, game_url in enumerate(games_urls): #[:DEBUG_GAMES_MAX]):
        print(f"Обработка URL игры: {game_url}")
        response = requests.get(game_url)
        if response.status_code != 200:
            print(f"* Ошибка при получении страницы игры: `{response.status_code}`")
            continue

        title_match = re.search(r'<title>([^<]+)</title>', response.text)
        if title_match:
            title = f"{i:03d}_" + re.sub(r'[^\w_]', '-', re.sub(r'\s+', '_', title_match.group(1).strip()))
        else:
            title = f"{i:03d}_" + "Unknown"

        output_file_path = os.path.join(OUTPUT_DIR, jam_path, f"{title}.html")
        with open(output_file_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
            print(f"Сохранено в `{output_file_path}`")
        jam_games_file.write(f"{output_file_path} {game_url}\n")

    jam_games_file.close()

# 2. Извлекаем данные со всех страниц игр
2.1 Используем уже скачанные HTML-страницы игр геймджема, чтобы извлечь данные о каждой игре

In [38]:
jam_games_file_path = os.path.join(OUTPUT_DIR, jam_path, "jam_games.txt")
jam_games_file = open(jam_games_file_path, 'r', encoding='utf-8')
print(f"Всего игр в файле `{jam_games_file_path}`: {len(jam_games_file.readlines())}")

Всего игр в файле `output/myindie-game-jam-level-9\jam_games.txt`: 68


2.2 Собираем оценки и отзывы для каждой игры из скачанных файлов

In [ ]:
all_judges_reviews = []  # без разделения по играм
all_participant_reviews = []  # без разделения по играм
all_users_reviews = []  # без разделения по играм
game_reviews = {}  # отзывы для каждой игры, ключ - alias игры, значение - словарь с отзывами по типам

jam_games_file.seek(0)

for line in jam_games_file:
    output_file_path, game_url = line.strip().split(' ', 1)
    alias = game_url.split('/')[-1]
    print(f"Обработка: {game_url}")
    with open(output_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    match = re.search(r'id=\"__NUXT_DATA__\">([^<]+)</script>', html_content)
    if not match:
        print(f"* Не найден __NUXT_DATA__ в `{output_file_path}`\n")
        continue

    json_data = json.loads(match.group(1))
    unflattened = unflatten_nuxt_data(json_data)
    if not unflattened:
        print(f"* Не удалось развернуть данные Nuxt.js в `{output_file_path}`\n")
        continue

    data_section = unflattened.get('data', [])
    if not data_section:
        print(f"* Не найден раздел 'data' в развернутых данных Nuxt.js в `{output_file_path}`\n")
        continue

    # Берем второй элемент списка (индекс 1) — там словарь с результатами
    if isinstance(data_section, list) and len(data_section) > 1:
        payload_container = data_section[1]
        if not payload_container:
            print(f"* Пустой контейнер в `{output_file_path}`\n")
            continue

        # В словаре берем первый ключ (game<alias>)
        if isinstance(payload_container, dict) and payload_container:
            second_key = list(payload_container.keys())[0]
            games = payload_container[second_key]
            if not games:
                print(f"* Пустой объект игры в `{output_file_path}`\n")
                continue

            # Извлекаем отзывы из game_payload['data']['reviews']
            if isinstance(games, dict):
                inner_data = games.get('data', {})
                if not inner_data:
                    print(f"* Пустой объект 'data' в `{output_file_path}`\n")
                    continue

                reviews = inner_data.get('reviews', [])
                if not reviews:
                    print(f"* Не найдено отзывов в `{output_file_path}`\n")
                    continue
                # print(json.dumps(reviews, ensure_ascii=False, indent=2))

                judges = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'judge' and r.get('user') is not None]
                all_judges_reviews.extend(judges)
                participants = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'jam-participant' and r.get('user') is not None]
                all_participant_reviews.extend(participants)
                users = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'user' and r.get('user') is not None]
                all_users_reviews.extend(users)

                game_reviews[alias] = {
                    'judges': judges,
                    'participants': participants,
                    'users': users,
                }
                print(f"Судейских отзывов: {len(judges)} - {', '.join([r.get('user', {}).get('username') for r in judges])}")
                print(f"Отзывы участников: {len(participants)} - {', '.join([r.get('user', {}).get('username') for r in participants])}")
                print(f"Отзывы пользователей: {len(users)} - {', '.join([r.get('user', {}).get('username') for r in users])}\n")

print(f"\nВсего судейских отзывов на джеме: {len(all_judges_reviews)}")
print(f"Всего отзывов участников на джеме: {len(all_participant_reviews)}")
print(f"Всего отзывов пользователей на джеме: {len(all_users_reviews)}")

Обработка: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
Судейских отзывов: 3 - PoliKhai, Saley, DmitriySklyarov
Отзывы участников: 11 - pafan, SemenLisenko, GhosDist, Ivancorej, KTD, chepuxxx, ArcasHH, Skyline_rozan, Tafari, Crp, CreatorLynx
Отзывы пользователей: 5 - AMONGAS, Waffle, MrGreen, dkotl, tenko

Обработка: https://myindie.ru/games/game/trash-under-ground
Судейских отзывов: 0 - 
Отзывы участников: 9 - GhosDist, KTD, ArcasHH, pafan, SemenLisenko, Tafari, GamerGrapes, Skyline_rozan, buZZi6X
Отзывы пользователей: 0 - 

Обработка: https://myindie.ru/games/game/cult-indie
Судейских отзывов: 4 - DmitriySklyarov, Saley, HelenAllienPoe, PoliKhai
Отзывы участников: 15 - JayxK, phanopera, GhosDist, pafan, Skyline_rozan, Jil, SemenLisenko, peped, Tafari, kafel, epigrodel, BobrikDobrik, Zarya61, KTD, Lacost
Отзывы пользователей: 9 - UnrealQW, Markus_Glevera, ChiFu, GrogShader, eturnercus, ignis, molodoy_venerez, niconikita, Rramad

Обработка: https://myindie.ru/games/ga

In [40]:
# game_reviews

In [41]:
jam_games_file.close()

# 3. Обработка данных и анализ

In [54]:
# print(json.dumps(all_judges_reviews, ensure_ascii=False, indent=2))
# print(json.dumps(all_participant_reviews, ensure_ascii=False, indent=2))
# print(json.dumps(all_users_reviews, ensure_ascii=False, indent=2))

3.1 Обрабатываем все судейские отзывы и оценки для каждой игры

In [ ]:
# Словари для хранения данных судей
judges_dict = {} # userId -> {"username": username, "reviews": []}

for review in all_judges_reviews:
    if review.get('reviewerType') == 'judge':
        user_id = review.get('userId')
        username = review.get('user', {}).get('username', 'Unknown')

        if user_id not in judges_dict:
            judges_dict[user_id] = {
                "username": username,
                "reviews": []
            }

        judges_dict[user_id]["reviews"].append(review)

judges_dict = dict(sorted(judges_dict.items(), key=lambda item: len(item[1]['reviews']), reverse=True))

print(f"Количество уникальных судей: {len(judges_dict)}")
for user_id, info in judges_dict.items():
    print(f"Судья: {info['username']} (ID: {user_id}) - Обзоров: {len(info['reviews'])}")

# Финальный словарь с идентификацией по username
judges_reviews_by_username = {info['username']: info['reviews'] for info in judges_dict.values()}

Количество уникальных судей: 4
Судья: PoliKhai (ID: c739f699-9424-4039-9c0b-8b26a6f0f1c2) - Обзоров: 41
Судья: DmitriySklyarov (ID: 9b681ea6-e7cb-489e-bae7-0e07607f93d4) - Обзоров: 33
Судья: Saley (ID: 6ddb6287-691d-41f2-a7db-5df11f84c1cf) - Обзоров: 32
Судья: HelenAllienPoe (ID: db51c67e-9d3c-4e25-a589-ec045a0cdb62) - Обзоров: 14


In [44]:
# print(json.dumps(judges_reviews_by_username, ensure_ascii=False, indent=2))

3.1.1 Обрабатываем все отзывы участников джема и оценки для каждой игры

In [ ]:
# Словари для хранения данных участников джема
participants_dict = {} # userId -> {"username": username, "reviews": []}

for review in all_participant_reviews:
    if review.get('reviewerType') == 'jam-participant':
        user_id = review.get('userId')
        username = review.get('user', {}).get('username', 'Unknown')

        if user_id not in participants_dict:
            participants_dict[user_id] = {
                "username": username,
                "reviews": []
            }

        participants_dict[user_id]["reviews"].append(review)

participants_dict = dict(sorted(participants_dict.items(), key=lambda item: len(item[1]['reviews']), reverse=True))

print(f"Количество уникальных участников джема: {len(participants_dict)}")
for user_id, info in participants_dict.items():
    print(f"Участник джема: {info['username']} (ID: {user_id}) - Обзоров: {len(info['reviews'])}")

# Финальный словарь с идентификацией по username
participants_reviews_by_username = {info['username']: info['reviews'] for info in participants_dict.values()}

Количество уникальных участников джема: 43
Участник джема: pafan (ID: 89fb48cf-fad9-414e-ab0b-4f15a459ea54) - Обзоров: 66
Участник джема: SemenLisenko (ID: c0d4dd24-f39f-4942-84ba-be017ca1a0d3) - Обзоров: 66
Участник джема: KTD (ID: ee1c15e2-9f72-4aff-9849-1334ac6c4111) - Обзоров: 49
Участник джема: ArcasHH (ID: 57ecbb87-3217-4cc9-9f07-efbc378e92a2) - Обзоров: 32
Участник джема: Lacost (ID: beca03f4-f109-49ba-9451-24844c558782) - Обзоров: 24
Участник джема: Skyline_rozan (ID: eb0a4a15-8b6c-4d1b-9f23-27ff97343ee2) - Обзоров: 23
Участник джема: buZZi6X (ID: 8cb9624c-fe85-4fec-aeba-6be51d927fd9) - Обзоров: 23
Участник джема: JayxK (ID: 21cf988c-1d47-4615-8c5b-f81866ea19a9) - Обзоров: 23
Участник джема: GamerGrapes (ID: acee7cf0-0228-4ff1-bef1-2d12a03ad335) - Обзоров: 21
Участник джема: phanopera (ID: 66e8ace1-adf9-4d3d-8354-8ec0aab4076c) - Обзоров: 20
Участник джема: sasha (ID: c75bcb7b-9f3f-479a-a56a-2bb41296f5f4) - Обзоров: 19
Участник джема: GhosDist (ID: 1d0479d0-c97e-4a69-8e3f-caf34b

In [46]:
# print(json.dumps(participants_reviews_by_username, ensure_ascii=False, indent=2))

3.2 Собираем статистику по каждому судье: средний балл, количество оценок, количество отзывов

In [68]:
judges_stats = []

for username, reviews in judges_reviews_by_username.items():
    if not reviews:
        print(f"* Судья {username} не имеет обзоров, пропускаем.")
        continue

    totals = { 'score': [] }
    for criteria in CRITERIAS:
        totals[criteria] = []

    for r in reviews:
        if 'score' in r:
            totals['score'].append(r['score'])

        c = r.get('criterias', {})
        for criteria in CRITERIAS:
            if criteria in c:
                totals[criteria].append(c[criteria])

    judge_row = {
        'username': username,
        'reviews_count': len(reviews)
    }

    for key, values in totals.items():
        judge_row[f'avg_{key}'] = round(sum(values) / len(values), 3) if values else 0

    judges_stats.append(judge_row)

df_judges_stats = pd.DataFrame(judges_stats)

# Вывод результата, отсортированного по среднему баллу
df_judges_stats.sort_values(by='avg_score', ascending=False)


,username,reviews_count,avg_score,avg_art,avg_sound,avg_theme,avg_gameplay,avg_narrative,avg_overall_impression
3,HelenAllienPoe,14,3.829,4.071,4.143,4.143,3.536,3.357,3.679
2,Saley,32,3.528,4.094,3.562,3.422,3.031,3.344,3.609
0,PoliKhai,41,3.305,3.451,3.207,3.695,2.829,3.305,3.268
1,DmitriySklyarov,33,3.115,3.788,3.212,2.833,2.788,2.970,3.015


3.2.1 Собираем статистику по каждому участнику джема: средний балл, количество оценок, количество отзывов

In [69]:
participants_stats = []

for username, reviews in participants_reviews_by_username.items():
    if not reviews:
        print(f"* Участник джема {username} не имеет обзоров, пропускаем.")
        continue

    totals = { 'score': [] }
    for criteria in CRITERIAS:
        totals[criteria] = []

    for r in reviews:
        if 'score' in r:
            totals['score'].append(r['score'])

        c = r.get('criterias', {})
        for criteria in CRITERIAS:
            if criteria in c:
                totals[criteria].append(c[criteria])

    participant_row = {
        'username': username,
        'reviews_count': len(reviews)
    }

    for key, values in totals.items():
        participant_row[f'avg_{key}'] = round(sum(values) / len(values), 3) if values else 0

    participants_stats.append(participant_row)

df_participants_stats = pd.DataFrame(participants_stats)

# Вывод результата, отсортированного по среднему баллу
df_participants_stats.sort_values(by='avg_score', ascending=False)

,username,reviews_count,avg_score,avg_art,avg_sound,avg_theme,avg_gameplay,avg_narrative,avg_overall_impression
36,chepuxxx,1,5.000,5.000,5.000,5.000,5.000,5.000,5.000
39,Veles,1,5.000,5.000,5.000,5.000,5.000,5.000,5.000
34,Bogila,2,4.500,5.000,5.000,5.000,3.250,5.000,3.750
20,BobrikDobrik,10,4.220,4.450,4.000,4.400,4.150,4.000,4.300
25,Jil,6,4.150,4.250,4.250,3.917,4.333,3.750,4.333
42,bRokesy,1,4.000,3.500,4.000,4.500,3.500,4.000,4.500
3,ArcasHH,32,3.928,4.094,4.188,4.156,3.406,4.250,3.453
33,AlexeySergeevich,2,3.900,4.000,4.000,4.750,3.250,3.750,3.500
16,Deleted user 33d3f881,11,3.836,3.545,3.409,4.545,3.455,3.727,4.273
26,Dionis_369,6,3.800,3.750,4.250,4.500,3.083,3.500,3.750


3.3 Cтатистический анализ: средние оценки, медианы, стандартные отклонения, распределения оценок и длины отзывов

In [70]:
judge_detailed_stats = []

for username, reviews in judges_reviews_by_username.items():
    judge_precise_scores = []
    judge_review_lengths = []

    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]

        if crit_values:
            game_score = sum(crit_values) / len(crit_values)
            judge_precise_scores.append(game_score)

        text = r.get('reviewText', '')
        if text:
            clean_text = re.sub(r'<[^>]+>', '', text)
            judge_review_lengths.append(len(clean_text))

    stats = {
        'username': username,
        'min_score': np.min(judge_precise_scores) if judge_precise_scores else 0.0,
        'max_score': np.max(judge_precise_scores) if judge_precise_scores else 0.0,
        'median_score': np.median(judge_precise_scores) if judge_precise_scores else 0.0,
        'std_score': np.std(judge_precise_scores) if judge_precise_scores else 0.0,
        'avg_review_chars': int(np.mean(judge_review_lengths)) if judge_review_lengths else 0.0,
        'min_review_chars': np.min(judge_review_lengths) if judge_review_lengths else 0,
        'max_review_chars': np.max(judge_review_lengths) if judge_review_lengths else 0
    }
    judge_detailed_stats.append(stats)

df_judge_detailed = pd.DataFrame(judge_detailed_stats)

df_judge_final_stats = pd.merge(df_judges_stats, df_judge_detailed, on='username')

cols = ['username', 'reviews_count', 'avg_score', 'median_score', 'std_score', 'min_score', 'max_score', 'avg_review_chars', 'min_review_chars', 'max_review_chars']
df_judge_final_stats[cols].sort_values(by='avg_score', ascending=False)

,username,reviews_count,avg_score,median_score,std_score,min_score,max_score,avg_review_chars,min_review_chars,max_review_chars
3,HelenAllienPoe,14,3.829,3.917,0.485,2.917,4.583,572,72,1038
2,Saley,32,3.528,3.500,0.671,2.083,4.667,777,411,2139
0,PoliKhai,41,3.305,3.417,0.783,1.083,4.667,532,69,1350
1,DmitriySklyarov,33,3.115,3.083,0.666,1.500,4.250,591,292,1458


3.3.1 Cтатистический анализ: средние оценки, медианы, стандартные отклонения, распределения оценок и длины отзывов

In [71]:
participants_detailed_stats = []

for username, reviews in participants_reviews_by_username.items():
    participant_precise_scores = []
    participant_review_lengths = []

    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]

        if crit_values:
            game_score = sum(crit_values) / len(crit_values)
            participant_precise_scores.append(game_score)

        text = r.get('reviewText', '')
        if text:
            clean_text = re.sub(r'<[^>]+>', '', text)
            participant_review_lengths.append(len(clean_text))

    stats = {
        'username': username,
        'min_score': np.min(participant_precise_scores) if participant_precise_scores else 0.0,
        'max_score': np.max(participant_precise_scores) if participant_precise_scores else 0.0,
        'median_score': np.median(participant_precise_scores) if participant_precise_scores else 0.0,
        'std_score': np.std(participant_precise_scores) if participant_precise_scores else 0.0,
        'avg_review_chars': int(np.mean(participant_review_lengths)) if participant_review_lengths else 0.0,
        'min_review_chars': np.min(participant_review_lengths) if participant_review_lengths else 0,
        'max_review_chars': np.max(participant_review_lengths) if participant_review_lengths else 0
    }
    participants_detailed_stats.append(stats)

df_participants_detailed = pd.DataFrame(participants_detailed_stats)

df_participants_final_stats = pd.merge(df_participants_stats, df_participants_detailed, on='username')

cols = ['username', 'reviews_count', 'avg_score', 'median_score', 'std_score', 'min_score', 'max_score', 'avg_review_chars', 'min_review_chars', 'max_review_chars']
df_participants_final_stats[cols].sort_values(by='avg_score', ascending=False)

,username,reviews_count,avg_score,median_score,std_score,min_score,max_score,avg_review_chars,min_review_chars,max_review_chars
36,chepuxxx,1,5.000,5.000,0.000,5.000,5.000,344.000,344,344
39,Veles,1,5.000,5.000,0.000,5.000,5.000,147.000,147,147
34,Bogila,2,4.500,4.500,0.500,4.000,5.000,212.000,152,272
20,BobrikDobrik,10,4.220,4.542,0.684,2.667,4.917,551.000,197,948
25,Jil,6,4.150,4.167,0.404,3.333,4.583,458.000,190,672
42,bRokesy,1,4.000,4.000,0.000,4.000,4.000,330.000,330,330
3,ArcasHH,32,3.928,4.000,0.536,2.583,4.667,615.000,114,1184
33,AlexeySergeevich,2,3.900,3.875,0.625,3.250,4.500,133.000,73,193
16,Deleted user 33d3f881,11,3.836,4.167,0.809,1.833,4.667,0.000,0,0
26,Dionis_369,6,3.800,3.750,0.627,3.000,5.000,240.000,0,478


3.4 Больше статистики судей:
* strictness_index (индекс "строгости"): Отрицательный = строгий, Положительный = добрый

Строгость считается от среднего значения оценок, выставленных судьями. Если судья ставит в среднем низкие оценки, то он считается строгим, если высокие -- добрым. Этот индекс может быть полезен для анализа "везучести" игр.

In [72]:
judges_deep_insights = []

all_judges_precise_scores = []
for reviews in judges_reviews_by_username.values():
    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]
        if crit_values:
            all_judges_precise_scores.append(sum(crit_values) / len(crit_values))

judges_global_avg = np.mean(all_judges_precise_scores) if all_judges_precise_scores else 0.0

for username, reviews in judges_reviews_by_username.items():
    if not reviews: continue

    judge_precise_scores = []
    judge_review_dates = []

    # Статистика по критериям для этого судьи
    criteria_totals = {k: [] for k in CRITERIAS}

    for r in reviews:
        c = r.get('criterias', {})
        cv = [c[k] for k in CRITERIAS if k in c]
        if cv:
            judge_precise_scores.append(sum(cv)/len(cv))

        for k in CRITERIAS:
            if k in c: criteria_totals[k].append(c[k])

        # Время
        dt_str = r.get('createdAt')
        if dt_str:
            dt = datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%S.%fZ")
            judge_review_dates.append(dt.date())

    # любимый и нелюбимый критерий (avg)
    crit_avgs = {k: np.mean(v) if v else 0 for k, v in criteria_totals.items()}
    best_crit = max(crit_avgs, key=crit_avgs.get)
    worst_crit = min(crit_avgs, key=crit_avgs.get)

    judge_avg = np.mean(judge_precise_scores) if judge_precise_scores else 0.0

    insight = {
        'username': username,
        'strictness_index': round(judge_avg - judges_global_avg, 3), # Отрицательный = строгий, Положительный = добрый
        'top_criteria': f"{best_crit} ({round(crit_avgs[best_crit], 2)})",
        'bottom_criteria': f"{worst_crit} ({round(crit_avgs[worst_crit], 2)})",
        'active_days': (max(judge_review_dates) - min(judge_review_dates)).days + 1 if judge_review_dates else 0,
        'reviews_per_day': round(len(reviews) / ((max(judge_review_dates) - min(judge_review_dates)).days + 1), 2) if judge_review_dates else 0
    }
    judges_deep_insights.append(insight)

df_judges_insights = pd.DataFrame(judges_deep_insights)

df_judges_insights.sort_values(by='strictness_index')


,username,strictness_index,top_criteria,bottom_criteria,active_days,reviews_per_day
1,DmitriySklyarov,-0.259,art (3.79),gameplay (2.79),7,4.710
0,PoliKhai,-0.067,theme (3.7),gameplay (2.83),3,13.670
2,Saley,0.151,art (4.09),gameplay (3.03),3,10.670
3,HelenAllienPoe,0.462,sound (4.14),narrative (3.36),4,3.500


3.4.1 Больше статистики участников джема:
- strictness_index: Отрицательный = строгий, Положительный = добрый

In [74]:
participants_deep_insights = []

all_participants_precise_scores = []
for reviews in participants_reviews_by_username.values():
    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]
        if crit_values:
            all_participants_precise_scores.append(sum(crit_values) / len(crit_values))

participants_global_avg = np.mean(all_participants_precise_scores) if all_participants_precise_scores else 0.0

for username, reviews in participants_reviews_by_username.items():
    if not reviews: continue

    participant_precise_scores = []
    participant_review_dates = []

    # Статистика по критериям для этого участника
    criteria_totals = {k: [] for k in CRITERIAS}

    for r in reviews:
        c = r.get('criterias', {})
        cv = [c[k] for k in CRITERIAS if k in c]
        if cv:
            participant_precise_scores.append(sum(cv)/len(cv))

        for k in CRITERIAS:
            if k in c: criteria_totals[k].append(c[k])

        # Время
        dt_str = r.get('createdAt')
        if dt_str:
            dt = datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%S.%fZ")
            participant_review_dates.append(dt.date())

    # любимый и нелюбимый критерий (avg)
    crit_avgs = {k: np.mean(v) if v else 0 for k, v in criteria_totals.items()}
    best_crit = max(crit_avgs, key=crit_avgs.get)
    worst_crit = min(crit_avgs, key=crit_avgs.get)

    participant_avg = np.mean(participant_precise_scores) if participant_precise_scores else 0.0

    insight = {
        'username': username,
        'strictness_index': round(participant_avg - participants_global_avg, 3), # Отрицательный = строгий, Положительный = добрый
        'top_criteria': f"{best_crit} ({round(crit_avgs[best_crit], 2)})",
        'bottom_criteria': f"{worst_crit} ({round(crit_avgs[worst_crit], 2)})",
        'active_days': (max(participant_review_dates) - min(participant_review_dates)).days + 1 if participant_review_dates else 0,
        'reviews_per_day': round(len(reviews) / ((max(participant_review_dates) - min(participant_review_dates)).days + 1), 2) if participant_review_dates else 0
    }
    participants_deep_insights.append(insight)

df_participants_insights = pd.DataFrame(participants_deep_insights)

df_participants_insights.sort_values(by='strictness_index')


,username,strictness_index,top_criteria,bottom_criteria,active_days,reviews_per_day
38,HenjTV,-2.227,art (1.0),art (1.0),1,1.000
40,Kosd,-1.893,sound (2.0),gameplay (1.0),1,1.000
14,Tafari,-1.004,overall_impression (2.47),theme (1.97),1,15.000
31,Trase,-0.949,art (3.17),narrative (1.33),1,3.000
24,CreatorLynx,-0.715,theme (3.64),gameplay (1.36),3,2.330
0,pafan,-0.709,theme (3.15),narrative (2.1),3,22.000
15,Zarya61,-0.625,theme (3.04),narrative (2.0),2,7.000
28,YuraBulba,-0.414,narrative (3.62),gameplay (2.0),4,1.000
32,yurimavt,-0.310,theme (4.0),gameplay (2.0),1,2.000
4,Lacost,-0.258,theme (3.88),gameplay (2.4),4,6.000


3.5 "Везучие" и "невезучие" игры

In [75]:
# Словари для быстрого поиска строгости
judge_strictness = df_judges_insights.set_index('username')['strictness_index'].to_dict()
participant_strictness = df_participants_insights.set_index('username')['strictness_index'].to_dict()

for alias, reviews in game_reviews.items():
    judges = reviews.get('judges', [])
    participants = reviews.get('participants', [])

    # Сбор данных по судьям
    j_info = []
    j_total_strictness = 0
    for r in judges:
        name = r.get('user', {}).get('username')
        strictness = judge_strictness.get(name, 0)
        j_info.append(f" {name} ({strictness})")
        j_total_strictness += strictness

    # Сбор данных по участникам
    p_info = []
    p_total_strictness = 0
    for r in participants:
        name = r.get('user', {}).get('username')
        strictness = participant_strictness.get(name, 0)
        p_info.append(f" {name} ({strictness})")
        p_total_strictness += strictness

    print(f"Игра: {alias}, URL: https://myindie.ru/games/game/{alias}")
    print(f"Судьи (строгость): {', '.join(j_info)}")
    print(f"Участники (строгость): {', '.join(p_info)}")
    print(f"Суммарная строгость судей: {round(j_total_strictness, 3)}")
    print(f"Суммарная строгость участников: {round(p_total_strictness, 3)}\n")

Игра: ultimatum-odna-noch-s-karachunom, URL: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
Судьи (строгость):  PoliKhai (-0.067),  Saley (0.151),  DmitriySklyarov (-0.259)
Участники (строгость):  pafan (-0.709),  SemenLisenko (0.276),  GhosDist (-0.143),  Ivancorej (0.393),  KTD (-0.121),  chepuxxx (1.773),  ArcasHH (0.698),  Skyline_rozan (-0.013),  Tafari (-1.004),  Crp (0.218),  CreatorLynx (-0.715)
Суммарная строгость судей: -0.175
Суммарная строгость участников: 0.653

Игра: trash-under-ground, URL: https://myindie.ru/games/game/trash-under-ground
Судьи (строгость): 
Участники (строгость):  GhosDist (-0.143),  KTD (-0.121),  ArcasHH (0.698),  pafan (-0.709),  SemenLisenko (0.276),  Tafari (-1.004),  GamerGrapes (0.031),  Skyline_rozan (-0.013),  buZZi6X (0.06)
Суммарная строгость судей: 0
Суммарная строгость участников: -0.925

Игра: cult-indie, URL: https://myindie.ru/games/game/cult-indie
Судьи (строгость):  DmitriySklyarov (-0.259),  Saley (0.151),  HelenAllien